# G33a — Temporal structure audit: ordered vs swap I↔II vs bag12 vs bag123 (5-fold CV, train+val, 10 seeds; test untouched)

Part of the **G33 campaign** with G33b (relational bias study). Both notebooks share the folds, seeds, preprocessing
(manifest version `G33-v1`) and the **R0 = `ordered`** artifacts, which are trained **only here** and exported for G33b.

**Where clip identity enters RoleNet (checked in the code).** Explicit: (1) the clip embedding on every token; (2) the
absent token, which has its own parameters per (role, clip); (3) the voice feature "cosine to clip III" (≈ 1 at clip III)
and its availability flag. Implicit, through the data: roles are defined from clip III (A = most frequent identity in
III, L = second), so A is almost always present in III and L exists only if seen in III. Conclusions are about
**explicit** clip identity only.

**Bag, defined operationally.** A model is a *bag over clip set S* iff (a) for every permutation π of the clips in S,
applied jointly to every clip-indexed input (faces of all roles, speech, scene, voice rows), f(πx) = f(x), checked
numerically on every evaluation row (max |Δ logit| < 1e-5; otherwise the arm is invalid), and (b) the known explicit
clip-identifying data feature is removed where S contains clip III (voice cosine to clip III and its flag, for bag123).
Construction: one shared clip embedding and one shared absent token (per role) for the clips in S.

| Arm | Training | Question |
|---|---|---|
| `ordered` (R0) | RoleNet as is | reference |
| `ordered + swap I↔II` (test time only) | the `ordered` model, clip I and II inputs exchanged, embedding positions fixed | does the trained model **rely** on I/II order? |
| `bag12` | I and II exchangeable, III distinct | is the order of the two past clips **useful** when the model is retrained? |
| `bag123` | all three clips exchangeable | is the **explicit** designation of current vs past useful? |
| swap on `bag12` / perms on `bag123` | validity check | must be 0 (invariance) |

**Selection, calibration, metric (fixed before running).** Checkpoint by inner-dev UAR (G14 rule, every arm). A
temperature T is fitted per arm × fold × seed on the inner-dev logits (same rows for all arms of a fold) and applied to
the held-out fold; the selection/calibration rows never overlap the evaluation rows. **Primary metric: mean of per-seed
temperature-calibrated NLL on the held-out folds.** Secondary: raw NLL, UAR, WAR. Intervals: 2,000 draws over seeds and
episodes, shared across arms.
* **D_reliance** = NLL(ordered + swap) − NLL(ordered) (same model, same T).
* **U_order** = NLL(bag12) − NLL(ordered).
* **U_boundary** = NLL(bag123) − NLL(bag12).
* Diagnostics only: mean total-variation change of the predicted distribution and the argmax flip rate under swap.

**Reading (fixed).**

| Result | Allowed statement about WHEN |
|---|---|
| U_order CI > 0 | historical **order** has utility (trajectory) |
| U_order CI ∋ 0, U_boundary CI > 0 | **explicit response-relative boundary information is useful**; no evidence for the order of past clips |
| both CIs ∋ 0 | history helps as **context** (G14 +2.13); no claim of temporal modelling |
| D_reliance CI > 0, U_order CI ∋ 0 | the trained model relies on order, but order is not shown to be needed |

"CI contains 0" means *not shown*, reported with the CI width (the smallest effect this design could detect). Removing
clip I alone cost only +0.30 UAR in G13, so an order effect may be small.

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER, N_INNER_DEV = 5, 5
SEEDS = [42, 123, 456, 7, 11, 19, 23, 31, 37, 43]    # as G14
LR, WEIGHT_DECAY = 1e-4, 1e-5                # B1 settings (only needed by the shared model cell)
FC_EPOCHS, PATIENCE, FC_BATCH = 50, 8, 32
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)     # G8b, unchanged
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None     # e.g. 6 for a quick smoke test

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True)
BASE = dict(readout='query', nq=1, q_readonly=False, role_emb=True, clip_emb=True, absent='token',
            drop=(), speech_parts=('text', 'audio', 'voice'), clips=(0, 1, 2), seed_offset=0)
EXPERIMENTS = [("RoleNet", 'role', FULL)]    # only used by the shared model cell's parameter print
N_BOOT = 2000
G33_VERSION = "G33-v1"                         # shared manifest / preprocessing version of the campaign
ARMS_A = {'ordered': 'ordered', 'bag12': 'bag12', 'bag123': 'bag123'}
INV_TOL = 1e-5

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = LRFZ = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
ZREC = torch.zeros(N, 3, N_REC, device=DEVICE)
YB, YA = T(DEV.yB.values), T(DEV.yA.values)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):                      # x [..., F, fin], m [..., F]
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNet(nn.Module):
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        if cfg['faces']:
            self.pool = FramePool(FDIM, d)
            self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
            self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        if cfg['ctx']:
            self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
            self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
            self.voice = nn.Linear(NVOICE, d)
            self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
            self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head = mk()
        self.head_face = mk() if cfg['aux'] and cfg['faces'] else None
        self.head_ctx = mk() if cfg['aux'] and cfg['ctx'] else None
        self.head_A = mk() if cfg['aux'] and cfg['faces'] and cfg['role'] else None

    def forward(self, ix, train=False):
        B, groups, aux = len(ix), [], {}
        if self.cfg['faces']:
            x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
            h, present = self.pool(x, m)                                      # [B, R, 3, d]
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
            ft = h.reshape(B, self.R * 3, -1)
            groups.append(ft)
            if self.head_face is not None:
                aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
            if self.head_A is not None:
                tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
                aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.cfg['ctx']:
            spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
            scn = self.scene(SCN[ix]) + self.ctx_role[1]
            ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)     # [B, 6, d]
            groups.append(ct)
            if self.head_ctx is not None:
                aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1)] + groups, 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop'] and len(groups) == 2:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = groups[0].shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


class B1Wrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False)

    def forward(self, ix, train=False):
        return self.f(CLIPIDX[ix], ZREC[ix]), {}


class B1FaceWrap(nn.Module):
    # B1 + the 60-d G6b-style role-face vector through a zero-initialised branch, trained jointly (the G7b design)
    def __init__(self, n_face=60, d=512):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, ix, train=False):
        b, clip_idx = self.f, CLIPIDX[ix]
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        tok = b.enc({k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(LRFZ[ix])), {}


HP = {'b1': dict(lr=LR, wd=WEIGHT_DECAY, epochs=FC_EPOCHS, patience=PATIENCE, batch=FC_BATCH),
      'role': dict(lr=RN['lr'], wd=RN['wd'], epochs=RN['epochs'], patience=RN['patience'], batch=RN['batch'])}
HP['b1face'] = HP['b1']
MAKE = {'b1': lambda cfg: B1Wrap(), 'b1face': lambda cfg: B1FaceWrap(), 'role': lambda cfg: RoleNet(cfg)}
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.2f}M" for n, k, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(kind, cfg, tr, dev, te, seed):
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## Shared G33 model, relation index, training and bootstrap helpers

In [ ]:
import json, hashlib, re
from scipy.optimize import minimize_scalar

# the eval fast path of nn.TransformerEncoderLayer mishandles per-head float attention masks (NaN once the relation
# biases are non-zero); it is switched off for every G33 arm, R0 included, so all arms run the same code path
if hasattr(torch.backends, 'mha') and hasattr(torch.backends.mha, 'set_fastpath_enabled'):
    torch.backends.mha.set_fastpath_enabled(False)
assert NVOICE == 9, "VOI layout changed: [sync r, sync sd] x 3 roles + [voice cos to clip III, voice ok, n identities]"
VOI_CLIP3_COLS = [6, 7]                        # voice cosine to clip III and its availability flag

# ---- folds (same construction as G8b-G32)
sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold_of_row = DEV.source_folder.map(FOLD).values
y_all = DEV.yB.values
src = DEV.source_folder.values
MANIFEST = {'version': G33_VERSION, 'seeds': SEEDS, 'n_outer': N_OUTER, 'n_inner_dev': N_INNER_DEV, 'RN': RN,
            'PCA_DIM': PCA_DIM, 'MAXF': MAXF, 'MAXF_POOL': MAXF_POOL, 'SAME_PERSON_COS': SAME_PERSON_COS,
            'DOMINANT_MIN_FRAC': DOMINANT_MIN_FRAC, 'VOICE_SAME_COS': VOICE_SAME_COS,
            'sample_ids': hashlib.sha1(','.join(DEV.sample_id.astype(str)).encode()).hexdigest(),
            'folds': hashlib.sha1(fold_of_row.astype(np.int64).tobytes()).hexdigest()}
print("fold sizes (MCIS):", load_, "| manifest:", {k: v for k, v in MANIFEST.items() if k in ('version', 'sample_ids', 'folds')})

# ---- token layout: 0 query | 1..9 faces (role r, clip k at 1 + 3r + k) | 10..12 speech (clip k) | 13..15 scene (clip k)
TOK = [('query', None, None)] + [('face', r, k) for r in range(3) for k in range(3)] + \
      [('speech', None, k) for k in range(3)] + [('scene', None, k) for k in range(3)]
NT = len(TOK)


def relation_index():
    # for attention i <- j (row i attends to column j); three factorial families
    T_ = np.zeros((NT, NT), int); P_ = np.zeros((NT, NT), int); E_ = np.zeros((NT, NT), int)
    for i, (ti, ri, ki) in enumerate(TOK):
        for j, (tj, rj, kj) in enumerate(TOK):
            q = ti == 'query' or tj == 'query'
            # TIME: 0 query-involved | 1 j same time as i | 2 j historically earlier than i | 3 j later than i
            T_[i, j] = 0 if q else (1 if kj == ki else (2 if kj < ki else 3))
            # PERSON: 0 same focal identity (A-A or L-L) | 1 A-L | 2 focal-O | 3 O-O (no identity claim) | 4 non-face pair
            if ti == 'face' and tj == 'face':
                if ri == rj and ri in (0, 1):
                    P_[i, j] = 0
                elif {ri, rj} == {0, 1}:
                    P_[i, j] = 1
                elif ri == 2 and rj == 2:
                    P_[i, j] = 3
                else:
                    P_[i, j] = 2
            else:
                P_[i, j] = 4
            # EVIDENCE: 0 face-speech | 1 face-scene | 2 speech-scene | 3 same evidence type | 4 other / query
            if q:
                E_[i, j] = 4
            elif ti == tj:
                E_[i, j] = 3
            else:
                E_[i, j] = {frozenset(('face', 'speech')): 0, frozenset(('face', 'scene')): 1,
                            frozenset(('speech', 'scene')): 2}[frozenset((ti, tj))]
    return {'time': T_, 'person': P_, 'evidence': E_}


REL = relation_index()
N_TYPES = {'time': 4, 'person': 5, 'evidence': 5}
NONQ = np.array([[TOK[i][0] != 'query' and TOK[j][0] != 'query' for j in range(NT)] for i in range(NT)])


def randomized_relations(seed):
    # per family: permute the labels of the ordered non-query pairs (marginal counts kept), query pairs unchanged
    rng = np.random.default_rng(seed)
    out = {}
    for f, M in REL.items():
        M = M.copy()
        M[NONQ] = rng.permutation(M[NONQ])
        out[f] = M
    return out


class RoleNetX(RoleNet):
    # RoleNet (G8b, all switches on) with (a) clip-identity mode ordered / bag12 / bag123, (b) optional factorial
    # relation biases on the attention logits, and (c) an optional permutation of the clip axis of every clip-indexed input.
    def __init__(self, clip='ordered', families=(), rel_index=None, d=RN['D']):
        super().__init__(FULL, d)
        self.clip_mode = clip
        self.register_buffer('cmap', torch.tensor({'ordered': [0, 1, 2], 'bag12': [0, 0, 2], 'bag123': [0, 0, 0]}[clip]))
        self.drop_clip3_voice = clip == 'bag123'
        self.families = tuple(families)
        if self.families:
            idx = rel_index or REL
            for f in self.families:
                self.register_buffer(f'rel_idx_{f}', torch.tensor(idx[f], dtype=torch.long))
            self.rel_b = nn.ParameterDict({f: nn.Parameter(torch.zeros(RN['layers'], RN['heads'], N_TYPES[f]))
                                           for f in self.families})          # zero init: starts as plain RoleNet

    def forward(self, ix, train=False, perm=None):
        B, aux = len(ix), {}
        p = torch.arange(3, device=DEVICE) if perm is None else torch.as_tensor(perm, device=DEVICE)
        x, m = FACE[ix][:, :, p], FMASK[ix][:, :, p]
        txt, aud, afd, scn, voi = TXT[ix][:, p], AUD[ix][:, p], AFD[ix][:, p], SCN[ix][:, p], VOI[ix][:, p]
        if self.drop_clip3_voice:
            voi = voi.clone(); voi[..., VOI_CLIP3_COLS] = 0
        h, present = self.pool(x, m)                                          # [B, 3, 3, d]
        absent = self.absent[:, self.cmap].unsqueeze(0).expand(B, -1, -1, -1)
        clip_e = self.clip_emb[self.cmap]
        h = torch.where(present.unsqueeze(-1), h, absent) + self.face_role[None, :, None] + clip_e[None, None]
        ft = h.reshape(B, 9, -1)
        aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
        tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
        aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        spk_ = self.text(txt) + self.audio(aud) * afd.unsqueeze(-1) + self.voice(voi) + self.ctx_role[0]
        sc_ = self.scene(scn) + self.ctx_role[1]
        ct = torch.cat([spk_ + clip_e, sc_ + clip_e], 1)                     # [B, 6, d]
        aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1), ft, ct], 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            valid[:, 1:10] &= ~drop_face.unsqueeze(1)
            valid[:, 10:] &= ~drop_ctx.unsqueeze(1)
        if not self.families:
            out = self.enc(toks, src_key_padding_mask=~valid)
        else:
            Hh = RN['heads']
            kmask = torch.zeros(B, 1, 1, NT, device=toks.device).masked_fill(~valid[:, None, None, :], -1e4)   # finite: no NaN grads
            out = toks
            for l, layer in enumerate(self.enc.layers):
                bias = sum(self.rel_b[f][l][:, getattr(self, f'rel_idx_{f}')] for f in self.families)   # [H, T, T]
                out = layer(out, src_mask=(bias.unsqueeze(0) + kmask).reshape(B * Hh, NT, NT))
        return self.head(out[:, 0]), aux


def logits_of(model, ix, perm=None, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(model(ix[i:i + bs], perm=perm)[0].float().cpu())
    return torch.cat(out).numpy().astype(np.float64)


def nll_logits(z, y):
    z = z - z.max(1, keepdims=True)
    return float((np.log(np.exp(z).sum(1)) - z[np.arange(len(y)), y]).mean())


def row_nll(z, y):
    z = z - z.max(-1, keepdims=True)
    return np.log(np.exp(z).sum(-1)) - np.take_along_axis(z, y[..., None], -1)[..., 0]


def fit_temperature(z, y):
    r = minimize_scalar(lambda lt: nll_logits(z / np.exp(lt), y), bounds=(np.log(0.05), np.log(20.0)), method='bounded')
    return float(np.exp(r.x))


def train_run(make, tr, dev, te, seed, ckpt_path):
    # checkpoint by inner-dev UAR (G14 rule); temperature fitted on the same inner-dev rows; held-out fold scored after
    seed_all(seed)
    hp = HP['role']
    model = make().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad, best_ep = -1, None, 0, -1
    for ep in range(hp['epochs']):
        model.train()
        perm_ = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm_), hp['batch']):
            j = perm_[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(logits_of(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad, best_ep = u, 0, ep
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    torch.save(best_state, ckpt_path)
    zd, zt = logits_of(model, dev), logits_of(model, te)
    return model, zd, zt, fit_temperature(zd, y_dev), best, best_ep


def fold_rows(f):
    tr_eps = [e for e in EPS if FOLD[e] != f]
    dev_eps = sorted(random.Random(100 + f).sample(tr_eps, N_INNER_DEV))
    trr = np.where(np.isin(src, tr_eps))[0]
    fit_rows = np.where(np.isin(src, tr_eps) & ~np.isin(src, dev_eps))[0]
    dev_rows = np.where(np.isin(src, dev_eps))[0]
    te_rows = np.where(fold_of_row == f)[0]
    return trr, fit_rows, dev_rows, te_rows


def boot_pairs(LOSS, pairs, n_boot=N_BOOT, seed=0):
    # mean of per-seed NLL; draws resample seeds and episodes, shared across all arms and pairs
    S = next(iter(LOSS.values())).shape[0]
    gidx = [np.where(src == e)[0] for e in np.unique(src)]
    rng = np.random.default_rng(seed)

    def q(sidx, w):
        nl = {a: float((L[sidx].mean(0) * w).sum() / w.sum()) for a, L in LOSS.items()}
        return {**{f'NLL {a}': v for a, v in nl.items()}, **{k: nl[x] - nl[y] for k, (x, y) in pairs.items()}}
    point = q(np.arange(S), np.ones(N))
    draws = []
    for _ in range(n_boot):
        w = np.zeros(N)
        for i in rng.integers(0, len(gidx), len(gidx)):
            w[gidx[i]] += 1
        draws.append(q(rng.integers(0, S, S), w))
    D = pd.DataFrame(draws)
    return point, {k: np.percentile(D[k], [2.5, 97.5]) for k in D}

## 5-fold episode cross-validation

In [ ]:
os.makedirs(f"{OUT_DIR}/g33_ckpt", exist_ok=True)
Z = {a: np.full((len(SEEDS), N, 7), np.nan) for a in list(ARMS_A) + ['ordered_swap']}
TEMP = {a: np.zeros((N_OUTER, len(SEEDS))) for a in ARMS_A}
DEVZ, DEVROWS, INV, log = {}, {}, [], []
t0 = time.time()
for f in range(N_OUTER):
    trr, fit_rows, dev_rows, te_rows = fold_rows(f)
    DEVROWS[f] = dev_rows
    FACE, POOL, var = build_face_tensors(sorted(set(DEV.iloc[trr][['clip1', 'clip2', 'clip3']].values.ravel())))
    print(f"fold {f}: train {len(fit_rows)} | early-stop {len(dev_rows)} | eval {len(te_rows)}", flush=True)
    tr, dev, te = T(fit_rows), T(dev_rows), T(te_rows)
    for a, mode in ARMS_A.items():
        DEVZ[(a, f)] = np.zeros((len(SEEDS), len(dev_rows), 7))
        for si, seed in enumerate(SEEDS):
            model, zd, zt, Tt, sel, ep = train_run(lambda: RoleNetX(clip=mode), tr, dev, te, seed + 1000 * f,
                                                   f"{OUT_DIR}/g33_ckpt/{a}_fold{f}_seed{seed}.pt")
            Z[a][si, te_rows], DEVZ[(a, f)][si], TEMP[a][f, si] = zt, zd, Tt
            if a == 'ordered':
                Z['ordered_swap'][si, te_rows] = logits_of(model, te, perm=[1, 0, 2])
            perms = {'bag12': [[1, 0, 2]], 'bag123': [[1, 0, 2], [2, 1, 0], [1, 2, 0]]}.get(a, [])
            for pm in perms:
                INV.append({'arm': a, 'fold': f, 'seed': seed, 'perm': str(pm),
                            'max_abs_dlogit': float(np.abs(logits_of(model, te, perm=pm) - zt).max())})
            yt = y_all[te_rows]
            log.append({'fold': f, 'arm': a, 'seed': seed, 'sel_UAR': sel, 'best_epoch': ep, 'T': Tt,
                        'NLL_raw': nll_logits(zt, yt), 'NLL_cal': nll_logits(zt / Tt, yt),
                        'UAR': war_uar(zt.argmax(1), yt, 7)[1]})
            print(f"fold {f} {a:<8} seed {seed:>3}: sel UAR {sel:5.2f} (epoch {ep}) | T {Tt:.2f} | NLL cal "
                  f"{log[-1]['NLL_cal']:.4f} | UAR {log[-1]['UAR']:5.2f} | {(time.time() - t0) / 60:.1f} min", flush=True)
            del model
            torch.cuda.empty_cache()

LOG, INVD = pd.DataFrame(log), pd.DataFrame(INV)
LOG.to_csv(f"{OUT_DIR}/g33a_fold_seed_log.csv", index=False)
INVD.to_csv(f"{OUT_DIR}/g33a_invariance.csv", index=False)
np.savez(f"{OUT_DIR}/g33a_artifacts.npz", manifest=json.dumps(MANIFEST), sample_id=DEV.sample_id.values,
         fold=fold_of_row, y=y_all, src=src, **{f"outer_{a}": v for a, v in Z.items()},
         **{f"temp_{a}": v for a, v in TEMP.items()}, **{f"dev_{a}_fold{f}": v for (a, f), v in DEVZ.items()},
         **{f"devrows_fold{f}": r for f, r in DEVROWS.items()})
print("saved g33a_artifacts.npz (R0 = 'ordered' artifacts for G33b), logs, and g33_ckpt/")

## Invariance checks, estimands and the fixed reading

In [ ]:
print("== invariance (bag definition, part a): max |Δ logit| under clip permutations ==")
VALID = {'ordered': True}
for a in ('bag12', 'bag123'):
    mx = INVD[INVD.arm == a].max_abs_dlogit.max()
    VALID[a] = bool(mx < INV_TOL)
    print(f"  {a:<7} max {mx:.2e} -> {'valid bag' if VALID[a] else 'NOT invariant: arm invalid for conclusions'}")

foldT = lambda a: TEMP[a][fold_of_row]                                    # [N, S] temperature of each row's model
CAL = {a: Z[a] / foldT(a).T[..., None] for a in ARMS_A}
CAL['ordered_swap'] = Z['ordered_swap'] / foldT('ordered').T[..., None]
LOSS = {a: row_nll(v, np.broadcast_to(y_all, v.shape[:2])) for a, v in CAL.items()}
LOSS_RAW = {a: row_nll(v, np.broadcast_to(y_all, v.shape[:2])) for a, v in Z.items()}
PAIRS = {'D_reliance = NLL(ordered+swap) - NLL(ordered)': ('ordered_swap', 'ordered'),
         'U_order = NLL(bag12) - NLL(ordered)': ('bag12', 'ordered'),
         'U_boundary = NLL(bag123) - NLL(bag12)': ('bag123', 'bag12')}
point, CI = boot_pairs(LOSS, PAIRS)
praw, CIraw = boot_pairs(LOSS_RAW, PAIRS, n_boot=500, seed=1)
print("\nmean per-seed calibrated NLL:", {a: round(point[f'NLL {a}'], 4) for a in LOSS})
print("temperature (median per arm):", {a: round(float(np.median(TEMP[a])), 2) for a in TEMP})
print("best epoch (mean per arm):", LOG.groupby('arm').best_epoch.mean().round(1).to_dict())
for k in PAIRS:
    print(f"  {k:<46} {point[k]:+.4f} [{CI[k][0]:+.4f}, {CI[k][1]:+.4f}] (CI width {CI[k][1] - CI[k][0]:.4f}) | raw NLL "
          f"{praw[k]:+.4f} [{CIraw[k][0]:+.4f}, {CIraw[k][1]:+.4f}]")

sm = lambda z: np.exp(z - z.max(-1, keepdims=True)) / np.exp(z - z.max(-1, keepdims=True)).sum(-1, keepdims=True)
P0, P1 = sm(CAL['ordered']), sm(CAL['ordered_swap'])
tv = 0.5 * np.abs(P0 - P1).sum(-1)
flip = (P0.argmax(-1) != P1.argmax(-1))
print(f"\nswap diagnostics (not decision metrics): mean total-variation change {tv.mean():.4f} "
      f"(95th percentile {np.percentile(tv, 95):.4f}) | argmax flip rate {flip.mean() * 100:.1f}%")
both12 = FMASK[:, :, :2].any(-1).any(1).all(-1).cpu().numpy()
for name_, m_ in (('faces in both clip I and II', both12), ('not both', ~both12)):
    if m_.any():
        print(f"  MCIS with {name_} ({m_.sum()}): TV {tv[:, m_].mean():.4f} | flips {flip[:, m_].mean() * 100:.1f}%")

lo = lambda k: CI[k][0]
ko, kb, kr = list(PAIRS)[1], list(PAIRS)[2], list(PAIRS)[0]
print("\n== G33a reading (fixed rules; explicit clip identity only) ==")
if not (VALID['bag12'] and VALID['bag123']):
    print("  at least one bag arm failed the invariance check: the corresponding statements below are not allowed")
if VALID['bag12'] and lo(ko) > 0:
    print("  WHEN: historical ORDER has utility (trajectory)")
elif VALID['bag123'] and lo(kb) > 0:
    print("  WHEN: explicit response-relative boundary information is useful; no evidence for the order of past clips")
else:
    print("  WHEN: history helps as context (G14); no evidence for temporal modelling (order or explicit boundary)")
if lo(kr) > 0 and not lo(ko) > 0:
    print("  note: the trained model relies on I/II order, but order is not shown to be needed")
SUM = pd.DataFrame([{'arm': a, 'NLL_cal': point[f'NLL {a}'], 'NLL_raw': praw[f'NLL {a}'],
                     'UAR_ensemble': war_uar(sm(Z[a]).mean(0).argmax(1), y_all, 7)[1],
                     'UAR_seed_mean': np.mean([war_uar(Z[a][s].argmax(1), y_all, 7)[1] for s in range(len(SEEDS))])}
                    for a in LOSS])
print(SUM.round(4).to_string(index=False))
SUM.to_csv(f"{OUT_DIR}/g33a_summary.csv", index=False)
pd.DataFrame({k: [point[k], CI[k][0], CI[k][1]] for k in point}, index=['point', 'lo', 'hi']).T.to_csv(
    f"{OUT_DIR}/g33a_estimands.csv")